In [1]:
! pip freeze

absl-py==2.1.0
accelerate==1.11.0
aiohappyeyeballs==2.6.1
aiohttp==3.13.1
aiosignal==1.4.0
annotated-types==0.7.0
anyio==4.11.0
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
arrow==1.3.0
asttokens==3.0.0
astunparse==1.6.3
async-lru==2.0.5
attrs==25.4.0
audioread==3.0.1
av==16.0.1
babel==2.17.0
beautifulsoup4==4.14.2
bitsandbytes==0.48.1
bleach==6.2.0
blinker==1.4
certifi==2024.7.4
cffi==2.0.0
charset-normalizer==3.3.2
click==8.3.0
cmake==3.25.0
coloredlogs==15.0.1
comm==0.2.3
contourpy==1.3.3
cryptography==3.4.8
ctranslate2==4.6.0
cycler==0.12.1
datasets==4.2.0
dbus-python==1.2.18
debugpy==1.8.17
decorator==5.2.1
defusedxml==0.7.1
dill==0.4.0
distro==1.7.0
evaluate==0.4.6
executing==2.2.1
fastapi==0.119.0
faster-whisper==1.2.0
fastjsonschema==2.21.2
ffmpeg-python==0.2.0
filelock==3.20.0
flatbuffers==24.3.25
fonttools==4.60.1
fqdn==1.5.1
frozenlist==1.8.0
fsspec==2025.9.0
future==1.0.0
gast==0.6.0
google-pasta==0.2.0
grpcio==1.64.1
h11==0.16.0
h5py==3.11.0
hf-xet==1.1.10
httpcore==1.

In [ ]:

from pathlib import Path
import json


from src.run_whisper_bls import transcribe_with_optional_bias

# beam hook関数は既存のBeam Seachの改良：
# ビームの結果だけでなく，途中の候補を意図的に参照し，目的のtokenと合致したらその累積確率を強制的にmaxにして以降の探索で参照させる役割
# 詳しくはTeamsの2段組論文の音声パートを見ると分かる
from src.beam_hook import InspectConfig

"実施者のみ"
WAV = "/root/MedWhisper/2025115cleandata/スマホ/1_川村先生_練習.m4a"
"実施者，協力者"
#WAV = "/root/MedWhisper/2025115cleandata/スマホ/3_川村先生_協力者_大野.m4a"
"実施者，協力者，AED"
WAV = "/root/MedWhisper/2025115cleandata/スマホ/4_川村先生_協力者_大場AED.m4a"

#WAV ="/root/MedWhisper/20241018/右後_2回目_川村先生.wav"
OUT = "/root/MedWhisper/out_whisper"
Path(OUT).mkdir(parents=True, exist_ok=True)


DOMAIN_TERMS = [
    # "傷","傷病者"
]


inspect_cfg = InspectConfig(
    enable_inspect=True,
    # beam の途中候補でtopいくつを残すか選択できる
    topk=5,
    # これらで指定した語句と途中の候補が合えば，強制的に確率を高める
    targets=["傷","周囲","体","胸"],
   # targets = []
   # どのくらい強制力を持たせるか
    force_bonus=50,
    lock_after_hit=True,
)

# inspect_cfg.ban_token_ids = [15553] この値は文字化けしたトークン　
  
r1 = transcribe_with_optional_bias(
    audio_path=WAV,
    out_dir=OUT,
    domain_terms=DOMAIN_TERMS,
    #initial_prompt="傷 体 胸 ",  # これは既存のイニシャルプロンプトです
    #use_bias=True,
    inspect_cfg=inspect_cfg, #提案手法を使わない場合，こちらをコメントアウトしてください
)

print(r1.get("text",""))

M25000 河村雄貴 これからBLSを開始します傷病者発見周囲は安全です 感染防御に配慮します大丈夫ですか 大丈夫ですか 大丈夫ですか 誰か誰か誰か来てくださいあなた119番通報してくださいあなたAEDを持ってきてください必ずここに戻ってきてください胸とお腹を見て呼吸の確認胸骨圧迫を開始します1、2、3、4、5、6、7、8、9、102、2、3、4、5、6、7、8、9、103、2、3、4、5、6、7、8、9、10AED持ってきましたあなたAED使えますか使えません胸骨圧迫を変わってください1、2の3で変わりましょうせーの1、2、31、2、3、4、5、6、7、8、9、102、2、3、4、5、6、7、8、9体表面よし体に触れないでください離れてください安全確認します私離れてますあなた離れてます体から離れてください体から離れてください体から離れてください体から離れてください体から離れてください体から離れてください体から離れてください体から離れてください体から離れてください体から離れてください体から離れてください体から離れてください体から離れてください体から離れてください1、2、3、4、5、6、7、8、9、10、2、2、3救急体です救急体の方この方、3分前に目の前で倒れるところを見ました胸骨圧迫をしてAEDで1体ショックをしています意識はまだ戻っていませんこの人の身元はわからないですがこの人の荷物は足元にありますので一緒に持っていってください引き継ぎます以上


In [ ]:
import re
from collections import Counter

def count_terms(text, terms):
    return {t: len(re.findall(re.escape(t), text)) for t in terms}

def basic_stats(text):
    return {
        "len": len(text),
        "chars": Counter(text).most_common(5),
    }

# 1) no-bias
r0 = transcribe_with_optional_bias(
    audio_path=WAV,
    out_dir=OUT,
    domain_terms=DOMAIN_TERMS,
    use_bias=False,
)

# 2) with-bias
r1 = transcribe_with_optional_bias(
    audio_path=WAV,
    out_dir=OUT,
    domain_terms=DOMAIN_TERMS,
    use_bias=True,
    inspect_cfg=inspect_cfg,
)

t0 = r0.get("text","")
t1 = r1.get("text","")

print("=== NO BIAS ===")
print(t0)
print("terms:", count_terms(t0, DOMAIN_TERMS))
print("stats:", basic_stats(t0))

print("\n=== WITH BIAS ===")
print(t1)
print("terms:", count_terms(t1, DOMAIN_TERMS))
print("stats:", basic_stats(t1))

print("\n=== DIFF ===")
print("same_text:", t0 == t1)
print("delta_len:", len(t1) - len(t0))
for term in DOMAIN_TERMS:
    print(term, count_terms(t1,[term])[term] - count_terms(t0,[term])[term])

In [ ]:
from whisper.tokenizer import get_tokenizer
tok = get_tokenizer(multilingual=True, task="transcribe")

print("timestamp_begin:", tok.timestamp_begin)
print("eot:", tok.eot)
print("15553 is timestamp?",
      tok.timestamp_begin <= 15553 < tok.eot)

In [ ]:
from whisper.tokenizer import get_tokenizer
tok = get_tokenizer(multilingual=True, task="transcribe")
print(tok.decode([15553]))

In [ ]:

p = Path(OUT)
cand = sorted([x for x in p.glob("**/*") if x.is_file() and x.suffix in (".jsonl",".json")])
print("\n=== OUT DIR artifacts ===")
for x in cand:
    print(str(x))

# 下記ファイルにてにてbeam探索過程の途中のtoken候補が見れる
inspect_path = Path(OUT) / "inspect.jsonl"
beamlog_path = Path(OUT) / "result_bias_beamlog.jsonl"  

print("\n=== Expected ===")
print("inspect:", inspect_path, "exists=", inspect_path.exists())
print("beamlog:", beamlog_path, "exists=", beamlog_path.exists())


WATCH_TARGETS = ["傷", "傷病者", "傷病者発見", " 発見", "傷病"]  # 必要に応じて追加

def summarize_inspect_hits(inspect_jsonl: Path, watch_targets=None, phase="pre_bias", max_show=50):
    if watch_targets is None:
        watch_targets = []
    if not inspect_jsonl.exists():
        print(f"[warn] not found: {inspect_jsonl}")
        return

    total = 0
    pre_lines = 0
    hit_rows = []

    with inspect_jsonl.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            total += 1
            try:
                rec = json.loads(line)
            except Exception:
                continue

            if rec.get("phase") == phase:
                pre_lines += 1

                if rec.get("hits_best"):
                    hb = rec["hits_best"]
                    # watch_targets に含まれる target が一つでもあれば hit扱い
                    keep = False
                    for it in hb:
                        t = it.get("target","")
                        if (not watch_targets) or (t in watch_targets):
                            keep = True
                    if keep:
                        hit_rows.append(rec)
                else:
            
                    hits = rec.get("hits") or []
                    keep = False
                    for it in hits:
                        t = it.get("target","")
                        if (not watch_targets) or (t in watch_targets):
                            keep = True
                    if keep:
                        hit_rows.append(rec)

    print(f"[info] lines={total}, {phase}_lines={pre_lines}")

    # steps->beams をまとめる
    step2beams = {}
    for rec in hit_rows:
        step = rec.get("step")
        beam = rec.get("beam")
        if step is None or beam is None:
            continue
        step2beams.setdefault(step, set()).add(beam)

    print("\n=== steps where WATCH appeared in hits ===")
    for step in sorted(step2beams.keys()):
        beams = sorted(step2beams[step])
        print(f"step={step}: beams={beams}")

    print("\n=== details (only where WATCH appeared) ===")
    shown = 0
    for rec in hit_rows:
        if shown >= max_show:
            break
        step = rec.get("step")
        beam = rec.get("beam")
        targets = rec.get("targets")

        if rec.get("hits_best"):
            for it in rec["hits_best"]:
                print(
                    f"step={step} beam={beam} target={it.get('target')} "
                    f"rank={it.get('rank')} lp={it.get('logprob')} gap={it.get('gap_to_top1')} "
                    f"decoded={it.get('decoded')}"
                )
        else:

            hits = rec.get("hits") or []
            if hits:
                it = hits[0]
                print(
                    f"step={step} beam={beam} watch_logprob={it.get('logprob')} "
                    f"target={it.get('target')} decoded={it.get('decoded')} targets={targets}"
                )
        shown += 1

    
    print("\n=== target hit summary (rough) ===")
    counts = {t: {"seen": 0, "hit": 0} for t in (watch_targets or [])}
    if watch_targets:
        with inspect_jsonl.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except Exception:
                    continue
                if rec.get("phase") != phase:
                    continue
                for t in watch_targets:
                    counts[t]["seen"] += 1
                # 改修版
                if rec.get("hits_best"):
                    hit_targets = set([it.get("target","") for it in rec["hits_best"]])
                    for t in watch_targets:
                        if t in hit_targets:
                            counts[t]["hit"] += 1
                else:
          
                    hit_targets = set([it.get("target","") for it in (rec.get("hits") or [])])
                    for t in watch_targets:
                        if t in hit_targets:
                            counts[t]["hit"] += 1

        for t, d in counts.items():
            print(f"{t}: seen={d['seen']}, hit={d['hit']}")


summarize_inspect_hits(inspect_path, watch_targets=WATCH_TARGETS, phase="pre_bias", max_show=50)


summarize_inspect_hits(inspect_path, watch_targets=WATCH_TARGETS, phase="post_bias", max_show=50)